[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeutscheAktuarvereinigung/.github/blob/main/traffic/traffic_analysis.ipynb)

# Traffic Analysis – DeutscheAktuarvereinigung

Dieses Notebook lädt die aktuellste GitHub-Traffic-CSV aus Google Drive und analysiert sie interaktiv mit Altair.

In [ ]:
# Setup: Bibliotheken importieren
import pandas as pd
import altair as alt
import glob
import os

In [ ]:
# Aktuellste Traffic-CSV automatisch ermitteln
DRIVE_BASE = '.'
csv_files = sorted(glob.glob(os.path.join(DRIVE_BASE, 'github_traffic_dav_*.csv')), reverse=True)

if not csv_files:
    raise FileNotFoundError(f'Keine github_traffic_dav_*.csv in {DRIVE_BASE} gefunden.')

OUTPUT_CSV_PATH = csv_files[0]
print(f'Verwende aktuellste CSV: {OUTPUT_CSV_PATH}')

In [ ]:
# CSV laden
try:
    df_from_csv = pd.read_csv(OUTPUT_CSV_PATH, sep=';')
    # Convert 'Datum' column to datetime objects
    df_from_csv['Datum'] = pd.to_datetime(df_from_csv['Datum'])
    print(f'CSV loaded successfully from: {OUTPUT_CSV_PATH}')
except FileNotFoundError:
    print(f'Error: The file {OUTPUT_CSV_PATH} was not found. Please ensure the data collection cell was executed and the file was saved.')
    raise

# Re-create df_clean for charting purposes, using the data loaded from CSV
df_clean_from_csv = df_from_csv[df_from_csv['Datum'] > (df_from_csv['Datum'].max() - pd.Timedelta(days=14))]

# Define metrics list
metrics = ['Views', 'Clones', 'Unique Views', 'Unique Clones']

display(df_clean_from_csv.head())
print('Metrics for charting:', metrics)

In [ ]:
import altair as alt

# Create a selection for the metric dropdown
metric_selection = alt.selection_point(
    fields=['Metric'],
    bind=alt.binding_select(options=metrics, name='Select Metric '),
    value=metrics[0],
    name='metric_param'
)

# Create a multi-selection for repositories
repository_selection = alt.selection_point(
    fields=['Repository'],
    bind=alt.binding_select(options=df_clean_from_csv['Repository'].unique().tolist(), name='Select Repository '),
    name='repo_select'
)

# Create the base chart
base = alt.Chart(df_clean_from_csv).properties(
    title='GitHub Traffic Trends by Repository'
).add_params(
    metric_selection,
    repository_selection
).transform_fold(
    metrics,
    as_=['Metric', 'Value']
).transform_filter(
    metric_selection
).transform_filter(
    repository_selection
)

# Create the line chart
chart = base.mark_line(point=True).encode(
    x=alt.X('Datum:T', title='Date'),
    y=alt.Y('Value:Q', title='Metric Value', stack=None),
    color=alt.Color('Repository:N', title='Repository'),
    tooltip=[
        alt.Tooltip('Repository:N'),
        alt.Tooltip('Datum:T', title='Date'),
        alt.Tooltip('Value:Q', title='Metric Count', format='.0f')
    ]
).interactive()

chart